#Lab: Delta Tables
##**Course 1, Week 3: Delta Lake & Workflows**
###Objectives
- Create and manage Delta tables
- Perform INSERT, UPDATE, and MERGE operations
- Use time travel to query historical data
- Understand schema enforcement and evolution

##Part 1: Create a Delta Table
###Create a Delta table for an inventory system

In [0]:
# Create an inventory DataFrame with columns:
# sku (string), product_name (string), category (string), quantity (int), unit_price (double)
# Include at least 6 products across 2+ categories
products = spark.createDataFrame([
    ('SKU1', 'Apple iPhone 14', 'Electronics', 100, 799.0),
    ('SKU2', 'Samsung Galaxy S23', 'Electronics', 150, 749.0),
    ('SKU3', 'Sony WH-1000XM5 Headphones', 'Electronics', 200, 399.0),
    ('SKU4', 'Dell XPS 13 Laptop', 'Electronics', 80, 999.0),
    ('SKU5', 'Nike Air Max 270', 'Footwear', 250, 150.0),
    ('SKU6', 'Adidas Ultraboost 22', 'Footwear', 300, 180.0),
    ('SKU7', 'Puma RS-X Sneakers', 'Footwear', 220, 130.0),
    ('SKU8', 'Levi’s 501 Original Jeans', 'Apparel', 350, 60.0),
    ('SKU9', 'Polo Ralph Lauren T-Shirt', 'Apparel', 400, 45.0),
    ('SKU10', 'Instant Pot Duo 7-in-1 Cooker', 'Home Appliances', 180, 120.0)
], ['sku', 'product_name', 'category', 'quantity', 'unit_price'])

# Write the DataFrame as a Delta table named "lab_inventory"
products.write.format("delta").mode("overwrite").saveAsTable("lab_inventory")


In [0]:
%sql
-- Verify the table was created
SELECT * FROM lab_inventory;

##Part 2: INSERT New Records
###Add new products to the inventory

In [0]:
# Create a DataFrame with 2 new products and INSERT (append) them
new_products = spark.createDataFrame([
    ('SKU11', 'Bose QuietComfort Earbuds', 'Electronics', 120, 249.0),
    ('SKU12', 'Garmin Forerunner 945 GPS Watch', 'Electronics', 90, 499.0)
], ['sku', 'product_name', 'category', 'quantity', 'unit_price'])

new_products.write.format("delta").mode("append").saveAsTable("lab_inventory")

In [0]:
%sql
-- Verify the insert
SELECT COUNT(*) AS total_products FROM lab_inventory;

##Part 3: UPDATE Records
###Update prices for a category

In [0]:
%sql
-- Increase all prices by 5% for one category of your choice
UPDATE lab_inventory SET unit_price = unit_price * 1.05 WHERE category = 'Electronics';
    
-- Verify the update
SELECT * FROM lab_inventory WHERE category = 'Electronics';      


##Part 4: MERGE (Upsert)
###Perform a MERGE operation to update existing and insert new products

In [0]:
%sql
-- Create a source view with updates and new records
-- Include at least 1 existing SKU (update) and 1 new SKU (insert)
CREATE OR REPLACE VIEW inventory_merge_source AS
SELECT 'SKU2' AS sku, 'Samsung Galaxy S23 Ultra' AS product_name, 'Electronics' AS category, 160 AS quantity, 799.0 AS unit_price
UNION ALL
SELECT 'SKU13', 'Dyson V11 Vacuum Cleaner', 'Home Appliances', 75, 599.0;

-- Write a MERGE statement
-- Match on SKU
-- WHEN MATCHED: update quantity and unit_price
-- WHEN NOT MATCHED: insert all columns
MERGE INTO lab_inventory AS target
USING inventory_merge_source AS source
ON target.sku = source.sku
WHEN MATCHED THEN
  UPDATE SET quantity = source.quantity, unit_price = source.unit_price
WHEN NOT MATCHED THEN
  INSERT (sku, product_name, category, quantity, unit_price)
  VALUES (source.sku, source.product_name, source.category, source.quantity, source.unit_price);

-- Verify the merge
SELECT * FROM lab_inventory WHERE sku = 'SKU2' OR sku = 'SKU13';

##Part 5: Time Travel
###Query previous versions of the table

In [0]:
%sql
-- View the full history of the table
DESCRIBE HISTORY lab_inventory;

-- Query the original version (version 0) and compare with current data
SELECT * FROM lab_inventory VERSION AS OF 0;
    
-- Write a query that shows price changes between version 0 and current
-- Join current with VERSION AS OF 0 on SKU
-- Show: sku, product_name, original_price, current_price, price_change
SELECT current.sku, current.product_name, original.unit_price AS original_price, current.unit_price AS current_price, current.unit_price - original.unit_price AS price_change
FROM lab_inventory current
JOIN lab_inventory VERSION AS OF 0 original
ON current.sku = original.sku;

##Part 6: Schema Enforcement
###Test that Delta Lake enforces the schema

In [0]:
# Try to insert a DataFrame with wrong schema (missing columns)
# This should FAIL - verify that Delta Lake catches the error
# Hint: Try creating a DataFrame with only 2-3 columns and appending to lab_inventory
new_products2 = spark.createDataFrame([
    ('SKU11', 'Bose QuietComfort Earbuds', 'Electronics', 120, 249.0),
    ('SKU12', 'Garmin Forerunner 945 GPS Watch', 'Electronics', 90, 499.0)
], ['sku', 'product_name'])

new_products2.write.format("delta").mode("append").saveAsTable("lab_inventory")


##Lab Validation and Cleanup

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Table exists
    try:
        df = spark.sql("SELECT * FROM lab_inventory")
        checks.append(("Delta table exists", True))
    except Exception:
        checks.append(("Delta table exists", False))
        df = None

    # Check 2: Table has data
    if df:
        checks.append(("Table has data", df.count() >= 6))

    # Check 3: Multiple versions exist (operations were performed)
    try:
        history = spark.sql("DESCRIBE HISTORY lab_inventory")
        checks.append(("Multiple versions (DML performed)", history.count() >= 3))
    except Exception:
        checks.append(("Multiple versions (DML performed)", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab_inventory")
except Exception:
    pass